# 01 - Cleaning the scraped listings

Input: `data/raw/item_details.csv`, 3,221 used-phone listings scraped from divar.ir (Tehran) in July 2024. All text is Persian, as it appears on the site.

Output: `data/cleaned_item_details.csv`, produced by `src.features.clean`. This notebook walks through *why* each cleaning step exists; the actual code lives in `src/features.py` so the trainer and the app use exactly the same rules.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.features import RAW_PATH, CLEAN_PATH, COLUMN_MAP, PRICE_MIN, PRICE_MAX, clean

pd.set_option("display.max_columns", 20)
raw = pd.read_csv(RAW_PATH)
print(raw.shape)
raw.head()

## What the raw data looks like

Every column is text. Numbers use Persian digits (`۲۵۶ گیگابایت` = 256 GB), prices have Persian thousands separators and the word "Toman", and the SIM-count field has three values including "۳ و بیشتر" (3 or more).

In [ ]:
print(raw.isna().sum(), "\n")
for col in ["تعداد سیم‌کارت", "حافظهٔ داخلی", "مقدار رم", "وضعیت", "اصالت برند"]:
    print(f"{COLUMN_MAP[col]}: {raw[col].value_counts(dropna=False).to_dict()}\n")

## Problems the original notebook missed

1. **Whitespace variants in colour.** `آبی` and `آبی ` (trailing space) were kept as two different classes; 37 distinct colour strings instead of 18.
2. **Duplicate rows.** The scraper re-read the first page after every "load more" click, so the same listing was saved more than once. 211 raw rows are exact duplicates.
3. **Placeholder prices.** On Divar a price of 1,000 or 1,111 Toman means "contact me". There is also an iPhone 4 listed at 48 billion Toman. An IQR filter on price cannot catch the low end, because its lower bound is negative.

In [ ]:
colours = raw["رنگ"].dropna().str.replace(r"[A-Za-z]+", "", regex=True)
colours = colours[colours.str.strip() != ""]
print("distinct colour strings after removing Latin words:", colours.nunique())
print("distinct after also stripping whitespace:", colours.str.strip().nunique())
print("exact duplicate rows:", raw.duplicated().sum())

In [ ]:
from src.features import parse_price_toman
prices = raw["قیمت"].map(parse_price_toman)
print("listings under 100,000 Toman:", (prices < 100_000).sum())
print("listings over 300,000,000 Toman:", (prices > 300_000_000).sum())
prices.describe().apply("{:,.0f}".format)

## Cleaning rules

`clean()` applies, in order: drop rows with a missing field, drop rows where a field is "not specified" (`مطرح نیست`), parse the Persian numbers and units, normalise colour text (strip Latin words and whitespace), drop exact duplicates, and keep only prices between 500,000 and 300,000,000 Toman. It also adds a `Brand` column (first token of "Brand and Model").

In [ ]:
df = clean(raw, verbose=True)
df.head()

In [ ]:
df.describe(include="all").T

In [ ]:
print("Brand:", df["Brand"].value_counts().to_dict(), "\n")
print("Color:", df["Color"].value_counts().to_dict())

In [ ]:
df.to_csv(CLEAN_PATH, index=False, lineterminator="\n")
print(f"wrote {len(df)} rows to {CLEAN_PATH}")